In [1]:
import os

def read_documents(folder_path):
    documents = {}

    for file in os.listdir(folder_path):
        if file.endswith(".txt"):
            file_path = os.path.join(folder_path, file)

            with open(file_path, "r", encoding="utf-8") as f:
                documents[file] = f.read()

    return documents



documents = read_documents("data")
documents

{'doc1.txt': "# Artificial Intelligence and Machine Learning\n\nArtificial Intelligence (AI) is one of the most significant technological developments of the twenty-first century. It refers to the ability of computer systems to perform tasks that traditionally require human intelligence, such as learning, reasoning, decision-making, problem-solving, understanding language, and recognizing images. AI has evolved from a theoretical concept into a practical technology that influences nearly every aspect of modern life. From smartphones and virtual assistants to healthcare systems and self-driving vehicles, artificial intelligence is transforming the way people work, communicate, and solve complex problems.\n\nMachine Learning (ML) is a major branch of artificial intelligence. Instead of being explicitly programmed with fixed rules, machine learning algorithms learn patterns from data and use those patterns to make predictions or decisions. The more relevant and high-quality data a machine

In [2]:
import nltk

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\DELL\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [3]:
import contractions
import re
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import emoji

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def normalize_text(text):

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Expand contractions (Optional)
    text = contractions.fix(text)

    # 3. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # 4. Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # 5. Remove emojis
    text = emoji.replace_emoji(text, replace='')

    # 6. Remove numbers
    text = re.sub(r'\d+', '', text)

    # 7. Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # 8. Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # 9. Tokenization
    tokens = word_tokenize(text)

    # 10. Remove stop words
    tokens = [word for word in tokens if word not in stop_words]

    # 11. Remove single-character words
    tokens = [word for word in tokens if len(word) > 1]

    # 12. Lemmatization
    tokens = [lemmatizer.lemmatize(word, pos = "v") for word in tokens]

    return " ".join(tokens)

In [4]:
normalized_documents = {}

for filename, text in documents.items():
    normalized_documents[filename] = normalize_text(text)

In [5]:
normalized_documents

{'doc1.txt': 'artificial intelligence machine learn artificial intelligence ai one significant technological developments twentyfirst century refer ability computer systems perform task traditionally require human intelligence learn reason decisionmaking problemsolving understand language recognize image ai evolve theoretical concept practical technology influence nearly every aspect modern life smartphones virtual assistants healthcare systems selfdriving vehicles artificial intelligence transform way people work communicate solve complex problems machine learn ml major branch artificial intelligence instead explicitly program fix rule machine learn algorithms learn pattern data use pattern make predictions decisions relevant highquality data machine learn model receive accurate predictions generally become ability improve experience make machine learn different traditional program every rule must manually write developers growth artificial intelligence fuel several important factor o

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(
    normalized_documents.values()
)

In [7]:
tfidf_matrix.shape

(4, 945)

In [8]:
print(tfidf_matrix.toarray())

[[0.06305201 0.         0.         ... 0.01701534 0.         0.        ]
 [0.         0.02522818 0.         ... 0.01610282 0.         0.01989019]
 [0.         0.         0.01988422 ... 0.         0.         0.01567695]
 [0.02068934 0.         0.         ... 0.03349961 0.02624181 0.        ]]


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

In [10]:
import pandas as pd

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=normalized_documents.keys(),
    columns=normalized_documents.keys()
)

print(similarity_df)

          doc1.txt  doc2.txt  doc3.txt  doc4.txt
doc1.txt  1.000000  0.866065  0.109920  0.868526
doc2.txt  0.866065  1.000000  0.130968  0.871630
doc3.txt  0.109920  0.130968  1.000000  0.123308
doc4.txt  0.868526  0.871630  0.123308  1.000000


In [11]:
threshold = 0.70  # 70% similarity

filenames = list(normalized_documents.keys())

print("Potential Plagiarism Cases:\n")

for i in range(len(filenames)):
    for j in range(i + 1, len(filenames)):   # Avoid duplicate comparisons
        similarity = similarity_matrix[i][j]

        if similarity >= threshold:
            print(f"{filenames[i]} <--> {filenames[j]}")
            print(f"Similarity: {similarity * 100:.2f}%")
            print("-" * 40)

Potential Plagiarism Cases:

doc1.txt <--> doc2.txt
Similarity: 86.61%
----------------------------------------
doc1.txt <--> doc4.txt
Similarity: 86.85%
----------------------------------------
doc2.txt <--> doc4.txt
Similarity: 87.16%
----------------------------------------
